<a href="https://colab.research.google.com/github/damas88/python-exercises/blob/main/offshore_wind_energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# --- 1. Data Loading and Initial Preparation ---

# Define file paths
# Ensure these CSV files are in the same directory as your script,
# or provide the full path to them.
power_data_file = '發電量月資料(11404).csv'
renew_data_file = '再生能源發電量月資料(11404).csv'

# Define a comprehensive column renaming map for consistency and clear English names
# Wind columns are temporarily named '_MWh' to indicate original unit before conversion.
initial_column_rename_map = {
    '日期(年/月)': 'Date_Month',
    '單位': 'Unit',
    '全國發電量_總計': 'Total_Generation_GWh',
    '全國發電量_抽蓄水力': 'Generation_PumpedHydro_GWh',
    '全國發電量_火力_合計': 'Generation_Thermal_Total_GWh',
    '全國發電量_火力_燃煤': 'Generation_Thermal_Coal_GWh',
    '全國發電量_火力_燃油': 'Generation_Thermal_Oil_GWh',
    '全國發電量_火力_燃氣': 'Generation_Thermal_Gas_GWh',
    '全國發電量_核能': 'Generation_Nuclear_GWh',
    '全國發電量_再生能源_合計': 'Generation_Renewable_Total_GWh',
    '全國發電量_再生能源_慣常水力': 'Generation_Renewable_Hydro_GWh',
    '全國發電量_再生能源_地熱': 'Generation_Renewable_Geothermal_GWh',
    '全國發電量_再生能源_太陽光電': 'Generation_Renewable_SolarPV_GWh',
    '全國發電量_再生能源_風力': 'Generation_Renewable_Wind_GWh',
    '全國發電量_再生能源_生質能': 'Generation_Renewable_Biomass_GWh',
    '全國發電量_再生能源_廢棄物': 'Generation_Renewable_Waste_GWh',
    '風力_陸域': 'Wind_Onshore_MWh',
    '風力_離岸': 'Wind_Offshore_MWh'
}

print("--- Data Loading ---")

# Determine columns to load for df_power, excluding wind-specific columns from renew file
power_cols_to_load = [col_orig for col_orig, col_new in initial_column_rename_map.items()
                      if col_new not in ['Wind_Onshore_MWh', 'Wind_Offshore_MWh']]
df_power = pd.read_csv(power_data_file, usecols=power_cols_to_load)
print(f"Loaded '{power_data_file}'")

# Load renewable energy data (specifically for wind)
renew_cols_to_load = ['日期(年/月)', '風力_陸域', '風力_離岸']
df_renew = pd.read_csv(renew_data_file, usecols=renew_cols_to_load)
print(f"Loaded '{renew_data_file}'")

In [ ]:
# --- 2. Data Cleaning & Transformation ---

print("\n--- Data Cleaning & Transformation ---")

# Apply unit conversion for wind power from MWh to GWh (1 GWh = 1000 MWh)
df_renew[['風力_陸域', '風力_離岸']] = df_renew[['風力_陸域', '風力_離岸']] / 1000.0
print("Converted wind power from MWh to GWh.")

# Rename columns in both DataFrames using the defined map before merging
df_power.rename(columns=initial_column_rename_map, inplace=True)
df_renew.rename(columns=initial_column_rename_map, inplace=True)
print("Renamed columns to consistent English names.")

# Merge the two DataFrames on the common 'Date_Month' column
df = pd.merge(df_power, df_renew, on='Date_Month', how='inner')
print("Merged power and renewable dataframes on 'Date_Month'.")

# Rename wind columns to reflect their final GWh unit after conversion
df.rename(columns={
    'Wind_Onshore_MWh': 'Wind_Onshore_GWh',
    'Wind_Offshore_MWh': 'Wind_Offshore_GWh'
}, inplace=True)
print("Updated wind column names to reflect GWh unit.")

# Convert 'Date_Month' to datetime objects and set as index
df['Date_Month'] = pd.to_datetime(df['Date_Month'], format='%Y%m')
df.set_index('Date_Month', inplace=True)
df.sort_index(inplace=True) # Ensure chronological order for time-series analysis
print("Converted 'Date_Month' to datetime index and sorted.")

# Drop the redundant 'Unit' column if it exists after renaming
if 'Unit' in df.columns:
    df.drop(columns=['Unit'], inplace=True)
    print("Dropped 'Unit' column (redundant unit info).")

# Ensure all relevant generation columns are numeric, coercing errors to NaN
# This is crucial for plotting and calculations.
generation_columns = [col for col in df.columns if col.endswith('_GWh')]
for col in generation_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print("Ensured all generation columns are numeric.")

# Fill any remaining missing values (e.g., from 'coerce' or original NaNs) for plotting
# Uses forward fill, then backward fill to handle gaps at start/end.
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)
print("Handled missing values with forward and backward fill.")

# Check for missing values after conversion and filling
print("\nMissing values after processing and filling:")
print(df.isnull().sum())
print("\nDataFrame head after all transformations:")
print(df.head())

In [ ]:
# --- 3. Overall Electricity Generation Analysis & Plotting Functions ---

def create_plot(dataframe, columns_to_plot, title, ylabel, plot_type='line', alpha=0.7, legend_title=None, line_color=None):
    """
    Creates a good-looking plot based on specified type (line or area).

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        columns_to_plot (str or list): Column(s) to plot.
        title (str): Title of the plot.
        ylabel (str): Label for the y-axis.
        plot_type (str): 'line' or 'area'.
        alpha (float): Transparency for area plots.
        legend_title (str, optional): Title for the plot legend. Defaults to None.
        line_color (str, optional): Specific color for a single line plot.
                                    Uses seaborn's palette for multiple lines/area plots.
    """
    plt.figure(figsize=(14, 7))

    # Plotting based on type
    if plot_type == 'line':
        if isinstance(columns_to_plot, str): # Single column
            dataframe[columns_to_plot].plot(ax=plt.gca(), color=line_color)
        else: # Multiple columns, will use seaborn's palette
            dataframe[columns_to_plot].plot(ax=plt.gca())
    elif plot_type == 'area':
        dataframe[columns_to_plot].plot.area(ax=plt.gca(), alpha=alpha)
    else:
        raise ValueError("plot_type must be 'line' or 'area'.")

    plt.title(title, fontsize=16)
    plt.ylabel(ylabel, fontsize=12)
    plt.xlabel('Date', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)

    # Show legend only if there are multiple columns or explicitly specified for a single column
    if (isinstance(columns_to_plot, list) and len(columns_to_plot) > 1) or legend_title:
        plt.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()

print("\n--- Generating General Energy Plots ---")

# Set plotting theme using seaborn for better aesthetics and consistent style
# Using a qualitative palette ('tab10') for distinct category colors
sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.autolayout'] = True # Automatically adjust subplot parameters for a tight layout


# Plot 1: Total National Electricity Generation
create_plot(
    df, 'Total_Generation_GWh',
    'Total National Electricity Generation (GWh)', 'Generation (GWh)',
    line_color='darkblue'
)

# Plot 2: National Electricity Generation Mix Over Time
major_sources = ['Generation_Thermal_Total_GWh', 'Generation_Nuclear_GWh', 'Generation_Renewable_Total_GWh']
create_plot(
    df, major_sources,
    'National Electricity Generation Mix Over Time', 'Generation (GWh)',
    plot_type='area',
    legend_title='Energy Source Type'
)

# Plot 3: Thermal Power Generation Breakdown
thermal_breakdown = ['Generation_Thermal_Coal_GWh', 'Generation_Thermal_Oil_GWh', 'Generation_Thermal_Gas_GWh']
create_plot(
    df, thermal_breakdown,
    'Thermal Power Generation Breakdown', 'Generation (GWh)',
    plot_type='line',
    legend_title='Thermal Source'
)

# Plot 4: Renewable Energy Generation Breakdown
renewable_sources_breakdown = [
    'Generation_Renewable_Hydro_GWh', 'Generation_Renewable_Geothermal_GWh',
    'Generation_Renewable_SolarPV_GWh', 'Generation_Renewable_Wind_GWh',
    'Generation_Renewable_Biomass_GWh', 'Generation_Renewable_Waste_GWh'
]
create_plot(
    df, renewable_sources_breakdown,
    'Renewable Energy Generation Breakdown', 'Generation (GWh)',
    plot_type='area',
    legend_title='Renewable Source'
)

# Plot 5: Onshore vs. Offshore Wind Power Generation
wind_power_cols = ['Wind_Onshore_GWh', 'Wind_Offshore_GWh']
create_plot(
    df, wind_power_cols,
    'Onshore vs. Offshore Wind Power Generation', 'Generation (GWh)',
    plot_type='line',
    legend_title='Wind Type'
)

In [ ]:
# --- 4. Deep Dive: Offshore Wind Energy Analysis (Focus on Seasonality) ---

print("\n--- Deep Dive: Offshore Wind Energy Analysis ---")

# Calculate key metrics for Offshore Wind
offshore_wind_data = df['Wind_Offshore_GWh']

# 1. Overall Growth Trend
print(f"Offshore Wind Generation (GWh) - First Record: {offshore_wind_data.iloc[0]:.2f} (on {offshore_wind_data.index[0].strftime('%Y-%m')})")
print(f"Offshore Wind Generation (GWh) - Last Record: {offshore_wind_data.iloc[-1]:.2f} (on {offshore_wind_data.index[-1].strftime('%Y-%m')})")

# Calculate Compound Annual Growth Rate (CAGR) for non-zero generation period
offshore_data_non_zero = offshore_wind_data[offshore_wind_data > 0]
if not offshore_data_non_zero.empty and len(offshore_data_non_zero) >= 2:
    start_date_cagr = offshore_data_non_zero.index[0]
    end_date_cagr = offshore_data_non_zero.index[-1]
    start_value_cagr = offshore_data_non_zero.iloc[0]
    end_value_cagr = offshore_data_non_zero.iloc[-1]
    num_years_cagr = (end_date_cagr - start_date_cagr).days / 365.25

    if start_value_cagr > 0 and num_years_cagr > 0:
        cagr = (end_value_cagr / start_value_cagr)**(1/num_years_cagr) - 1
        print(f"Approximate Compound Annual Growth Rate (CAGR) for Offshore Wind (from {start_date_cagr.year} to {end_date_cagr.year}): {cagr:.2%}")
    else:
        print("Cannot calculate CAGR meaningfully for Offshore Wind (start value is zero or insufficient time period with non-zero generation).")
else:
    print("Insufficient non-zero data for meaningful CAGR calculation for Offshore Wind.")


# 2. Offshore Wind's Share of Total Renewable Generation & Total Wind Generation & Total National Generation
df['Offshore_Wind_Share_of_Renewables'] = (df['Wind_Offshore_GWh'] / df['Generation_Renewable_Total_GWh']) * 100
df['Offshore_Wind_Share_of_Total_Wind'] = (df['Wind_Offshore_GWh'] / df['Generation_Renewable_Wind_GWh']) * 100
df['Offshore_Wind_Share_of_Total_Generation'] = (df['Wind_Offshore_GWh'] / df['Total_Generation_GWh']) * 100 # THIS IS THE CRUCIAL LINE FOR SEASONAL SHARE

# Replace NaN/inf shares with 0 if total generation is 0, to avoid division by zero issues in plots
for col_share in ['Offshore_Wind_Share_of_Renewables', 'Offshore_Wind_Share_of_Total_Wind', 'Offshore_Wind_Share_of_Total_Generation']:
    df[col_share].replace([float('inf'), -float('inf')], pd.NA, inplace=True)
    df[col_share].fillna(0, inplace=True)


# Plot: Offshore Wind's Share of Total Renewables over time
create_plot(
    df, 'Offshore_Wind_Share_of_Renewables',
    "Offshore Wind's Share of Total Renewable Generation (%)", 'Share (%)',
    line_color='purple',
    legend_title='Offshore Wind Share'
)

# Plot: Offshore Wind's Share of Total Wind Generation over time
create_plot(
    df, 'Offshore_Wind_Share_of_Total_Wind',
    "Offshore Wind's Share of Total Wind Generation (%)", 'Share (%)',
    line_color='teal',
    legend_title='Offshore Wind Share'
)


# --- Focus on Seasonal Controversy ---

# Define seasons
# Winter: Dec (12), Jan (1), Feb (2)
# Summer/Autumn (high consumption): Jun (6), Jul (7), Aug (8), Sep (9), Oct (10), Nov (11)
winter_months = [12, 1, 2]
summer_autumn_months = [6, 7, 8, 9, 10, 11]

df['Month'] = df.index.month

# Calculate monthly averages for Offshore Wind Generation (GWh)
monthly_avg_offshore_wind = df.groupby('Month')['Wind_Offshore_GWh'].mean()
# Correctly calculate monthly averages for Offshore Wind Share of Total National Generation (%)
monthly_avg_offshore_wind_share_total_gen = df.groupby('Month')['Offshore_Wind_Share_of_Total_Generation'].mean()

# Plot: Average Monthly Offshore Wind Generation (GWh) with seasonal highlighting
plt.figure(figsize=(14, 7))
ax = monthly_avg_offshore_wind.plot(kind='bar', color=sns.color_palette('viridis', 12))
plt.title('Average Monthly Offshore Wind Generation (GWh)', fontsize=16)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Generation (GWh)', fontsize=12)
plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Create custom legend handles for seasonal highlighting
legend_handles = []
# Using consistent colors from a chosen palette
winter_color = sns.color_palette("Greens")[2]
summer_autumn_color = sns.color_palette("Oranges")[2]
other_color = sns.color_palette('viridis', 12)[0] # A representative color for 'other'

green_patch = plt.matplotlib.patches.Patch(color=winter_color, label='Winter (Dec-Feb)')
orange_patch = plt.matplotlib.patches.Patch(color=summer_autumn_color, label='Summer/Autumn (Jun-Nov)')
legend_handles.extend([green_patch, orange_patch])


# Apply highlighting based on month
for i, month in enumerate(monthly_avg_offshore_wind.index):
    if month in winter_months:
        ax.patches[i].set_facecolor(winter_color)
    elif month in summer_autumn_months:
        ax.patches[i].set_facecolor(summer_autumn_color)

plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


# Plot: Average Monthly Offshore Wind Share of Total National Generation (%) with seasonal highlighting
plt.figure(figsize=(14, 7))
ax = monthly_avg_offshore_wind_share_total_gen.plot(kind='bar', color=sns.color_palette('viridis', 12))
plt.title('Average Monthly Offshore Wind Share of Total National Generation (%)', fontsize=16)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Share (%)', fontsize=12)
plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Apply highlighting based on month
for i, month in enumerate(monthly_avg_offshore_wind_share_total_gen.index):
    if month in winter_months:
        ax.patches[i].set_facecolor(winter_color)
    elif month in summer_autumn_months:
        ax.patches[i].set_facecolor(summer_autumn_color)

plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

# Quantitative comparison for controversy
winter_avg_gen = df[df['Month'].isin(winter_months)]['Wind_Offshore_GWh'].mean()
summer_autumn_avg_gen = df[df['Month'].isin(summer_autumn_months)]['Wind_Offshore_GWh'].mean()

# Corrected: Use 'Offshore_Wind_Share_of_Total_Generation' for share calculation
winter_avg_share = df[df['Month'].isin(winter_months)]['Offshore_Wind_Share_of_Total_Generation'].mean()
summer_autumn_avg_share = df[df['Month'].isin(summer_autumn_months)]['Offshore_Wind_Share_of_Total_Generation'].mean()

print("\n--- Seasonal Offshore Wind Averages (GWh) ---")
print(f"Average Offshore Wind Generation in Winter (Dec-Feb): {winter_avg_gen:.2f} GWh")
print(f"Average Offshore Wind Generation in Summer/Autumn (Jun-Nov): {summer_autumn_avg_gen:.2f} GWh")

print("\n--- Seasonal Offshore Wind Share of Total National Generation (%) ---")
print(f"Average Offshore Wind Share in Winter (Dec-Feb): {winter_avg_share:.2f}%")
print(f"Average Offshore Wind Share in Summer/Autumn (Jun-Nov): {summer_autumn_avg_share:.2f}%")



In [ ]:
# --- 5. Conclusion & Summary ---
print("\n--- Conclusion ---")
print("This analysis provides insights into Taiwan's electricity generation mix, with a specific focus on offshore wind energy's seasonality.")
print(f"- Overall offshore wind generation has grown significantly, with the latest recorded value at {offshore_wind_data.iloc[-1]:.2f} GWh.")
print("The controversy regarding offshore wind's seasonal contribution is well-supported by the data:")
print(f"- **High Winter Contribution:** Average offshore wind generation is considerably higher in winter months (Dec-Feb), averaging {winter_avg_gen:.2f} GWh. During this period, it contributes approximately {winter_avg_share:.2f}% to total national generation.")
print(f"- **Lower Summer/Autumn Contribution:** In contrast, during the high power consumption season of summer and autumn (Jun-Nov), average offshore wind generation drops to {summer_autumn_avg_gen:.2f} GWh, contributing only about {summer_autumn_avg_share:.2f}% to total national generation.")
print("This seasonal variability highlights the need for robust energy storage solutions or flexible backup power to ensure grid stability, particularly during peak demand periods when offshore wind output is lower. Understanding these seasonal patterns is crucial for effective energy planning and policy-making.")
print("\nFurther analysis could include forecasting, policy impact assessment, or efficiency studies.")